In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
import time

In [20]:
BATCH_SIZE = 1024
LEARNING_RATE = 0.001
EPOCHS = 100
RFF_COMPONENTS = 2048  # Higher = better RBF approximation, but slower
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


In [21]:
def load_and_preprocess():
    col_names = ["duration","protocol_type","service","flag","src_bytes",
        "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
        "logged_in","num_compromised","root_shell","su_attempted","num_root",
        "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
        "is_host_login","is_guest_login","count","srv_count","serror_rate",
        "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
        "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
        "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
        "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
        "dst_host_rerror_rate","dst_host_srv_rerror_rate","label", "difficulty"]

    print("Loading NSL-KDD files...")
    train_df = pd.read_csv('KDDTrain+.txt', names=col_names)
    test_df = pd.read_csv('KDDTest+.txt', names=col_names)

    print(train_df.head())

    # Drop 'difficulty'
    train_df.drop('difficulty', axis=1, inplace=True)
    test_df.drop('difficulty', axis=1, inplace=True)

    # Labels: Normal -> 0, Attack -> 1
    y_train = np.where(train_df['label'] == 'normal', 0, 1)
    y_test = np.where(test_df['label'] == 'normal', 0, 1)
    
    X_train = train_df.drop('label', axis=1)
    X_test = test_df.drop('label', axis=1)

    # Identify columns
    cat_cols = ["protocol_type", "service", "flag"]
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    # Pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )

    print("Fitting preprocessor...")
    X_train_enc = preprocessor.fit_transform(X_train).astype(np.float32)
    X_test_enc = preprocessor.transform(X_test).astype(np.float32)

    return X_train_enc, y_train, X_test_enc, y_test

In [22]:
class RBFSVM(nn.Module):
    def __init__(self, input_dim, rff_dim=2048, sigma=1.0):
        super(RBFSVM, self).__init__()
        
        # 1. Random Fourier Features Layer (Fixed, not trained)
        # We sample weights from a Gaussian distribution
        self.rff_weights = nn.Parameter(torch.randn(input_dim, rff_dim) / sigma, requires_grad=False)
        self.rff_bias = nn.Parameter(torch.rand(rff_dim) * 2 * np.pi, requires_grad=False)
        
        # 2. Linear SVM Layer
        # This learns the boundary in the high-dimensional space
        self.fc = nn.Linear(rff_dim, 1)
        
    def forward(self, x):
        # Apply RFF Mapping: cos(Wx + b)
        # This approximates the RBF kernel function
        x_mapped = torch.cos(x @ self.rff_weights + self.rff_bias)
        
        # Apply Linear Decision Boundary
        return self.fc(x_mapped)

In [23]:
X_train, y_train, X_test, y_test = load_and_preprocess()


y_train_svm = torch.tensor(np.where(y_train == 0, -1, 1), dtype=torch.float32).to(device)

X_train_tensor = torch.tensor(X_train).to(device)
X_test_tensor = torch.tensor(X_test).to(device)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_svm), 
                            batch_size=BATCH_SIZE, shuffle=True)

# Initialize RBF Model
input_dim = X_train.shape[1]
# Sigma controls the "width" of the RBF kernel. 
model = RBFSVM(input_dim, rff_dim=RFF_COMPONENTS, sigma=2.0).to(device)

# Optimizer
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=0.001)

print(f"\nStarting RBF SVM Training (Approximated via RFF)...")
start_time = time.time()

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X).squeeze()
        
        # Hinge Loss
        loss = torch.mean(torch.clamp(1 - batch_y * outputs, min=0))
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")

print(f"Training Time: {time.time() - start_time:.2f}s")

Loading NSL-KDD files...
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                    0.60                      

In [24]:
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor).squeeze()
    # Predictions: If output >= 0 -> Attack (1), else Normal (0)
    predictions = torch.where(outputs >= 0, 1, 0).cpu().numpy()

print("\n--- RBF SVM Evaluation Results ---")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(classification_report(y_test, predictions, target_names=['Normal', 'Attack']))


--- RBF SVM Evaluation Results ---
Accuracy: 0.7762
              precision    recall  f1-score   support

      Normal       0.66      0.98      0.79      9711
      Attack       0.97      0.63      0.76     12833

    accuracy                           0.78     22544
   macro avg       0.82      0.80      0.78     22544
weighted avg       0.84      0.78      0.77     22544

